# HDX State of Palestine — Data Pipeline
**WASH Center for Forced Displacement | Spring 2026**

This notebook reads, cleans, and merges 4 HDX Excel files into one master dataset.

**Sources:**
- Attacks on Health Care (2016–2026)
- Protection in Danger (2020–2026)
- Food Systems Incidents (2023–2026)
- Water Systems Incidents (2023–2026)

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

print('Libraries loaded.')

## Step 2: Load the Raw Excel Files
Each file is downloaded directly from HDX. Place them in the same folder as this notebook.

In [ ]:
health_raw = pd.read_excel('2016-2026-pse-attacks-on-health-care-incident-data.xlsx')
prot_raw   = pd.read_excel('2020-2026-pse-protection-in-danger-incident-data.xlsx')
food_raw   = pd.read_excel('2024-2026-pse-conflict-related-incidents-affecting-food-systems-incident-data.xlsx')
water_raw  = pd.read_excel('2025-2026-pse-conflict-related-incidents-affecting-water-systems-incident-data__1_.xlsx')

print(f'Health:     {health_raw.shape[0]} rows, {health_raw.shape[1]} columns')
print(f'Protection: {prot_raw.shape[0]} rows, {prot_raw.shape[1]} columns')
print(f'Food:       {food_raw.shape[0]} rows, {food_raw.shape[1]} columns')
print(f'Water:      {water_raw.shape[0]} rows, {water_raw.shape[1]} columns')

## Step 3: Preview the Raw Data
Before transforming anything, let's see what the raw files look like.

In [ ]:
print('=== HEALTH FILE COLUMNS ===')
print(list(health_raw.columns))
health_raw.head(3)

In [ ]:
print('=== WATER FILE COLUMNS ===')
print(list(water_raw.columns))
water_raw.head(3)

## Step 4: Define the Standardized Schema

The raw files have different column names and structures.
We define a helper function that extracts the shared base columns from each file.

In [ ]:
EXTRACTED_DATE = '2026-04-08'

def get_base(df, source_file, sind_col, event_mod_col='Date Event Modified'):
    """
    Extracts shared columns that exist across all 4 HDX files.
    Returns a standardized base DataFrame.
    """
    return pd.DataFrame({
        'data_source':               'HDX / Insecurity Insight (SHCC)',
        'source_file':               source_file,
        'source_version':            df[event_mod_col].astype(str).str[:10],
        'sind_event_id':             df[sind_col].astype(str),
        'date':                      df['Date'].astype(str).str[:10],
        'year':                      pd.to_datetime(df['Date']).dt.year,
        'admin_1':                   df['Admin 1'],
        'latitude':                  df['Latitude'],
        'longitude':                 df['Longitude'],
        'geo_precision':             df['Geo Precision'],
        'reported_perpetrator':      df['Reported Perpetrator'],
        'reported_perpetrator_name': df['Reported Perpetrator Name'],
        'weapon_used':               df['Weapon Carried/Used'],
    })

print('Base schema function defined.')

## Step 5: Unpivot (Melt) the Indicator Columns

The raw files store indicators as **wide format** — one column per indicator per row.

We convert to **long format** — one row per indicator per incident.
This makes it much easier to filter, merge, and analyze.

```
BEFORE (wide):                          AFTER (long):
incident | workers_killed | fac_damaged  incident | indicator        | value
001      | 2             | 1            001      | workers_killed   | 2
                                         001      | fac_damaged      | 1
```

In [ ]:
def melt_indicators(df, base, indicators, subgroup, notes_series, prefix):
    """
    Unpivots indicator columns into long format.
    Adds missingness flags for any blank/null values.
    """
    frames = []
    for col, (code, name, cat) in indicators.items():
        tmp = base.copy()
        tmp['indicator_code']      = code
        tmp['indicator_name']      = name
        tmp['indicator_category']  = cat
        tmp['population_subgroup'] = subgroup
        raw_val = df[col]
        if raw_val.dtype == bool:
            raw_val = raw_val.astype(float)  # convert True/False to 1/0
        tmp['value']             = raw_val.where(pd.notna(raw_val), other=None)
        tmp['unit']              = 'count'
        tmp['missingness_flag']  = np.where(pd.isna(df[col]), 'MISSING_AMBIGUOUS', 'NOT_MISSING')
        tmp['missingness_type']  = np.where(pd.isna(df[col]),
                                            'TYPE_C: Blank — unclear if zero, not recorded, or truly missing', '')
        tmp['notes']             = notes_series
        tmp['data_extracted_date'] = EXTRACTED_DATE
        frames.append(tmp)

    out = pd.concat(frames, ignore_index=True)
    out.insert(0, 'record_id', [f'{prefix}-{i+1:06d}' for i in range(len(out))])
    out.insert(7, 'date_precision', 'day')
    out.insert(11, 'geographic_unit_type', 'Admin 1 Region')
    return out

print('Melt function defined.')

## Step 6: Process Each File

### 6a. Health Care File

In [ ]:
health_base = get_base(health_raw, '2016-2026-pse-attacks-on-health-care', 'SiND Event ID')

health_indicators = {
    'Number of Attacks on Health Facilities Reporting Destruction': ('HDX_H_FAC_DESTR',  'Health Facilities — Destruction Reported',   'Healthcare Infrastructure'),
    'Number of Attacks on Health Facilities Reporting Damaged':     ('HDX_H_FAC_DAMAGE', 'Health Facilities — Damage Reported',         'Healthcare Infrastructure'),
    'Forceful Entry into Health Facility':                          ('HDX_H_FAC_ENTRY',  'Forceful Entry into Health Facility',         'Healthcare Infrastructure'),
    'Occupation of Health Facility':                                ('HDX_H_FAC_OCCUP',  'Occupation of Health Facility',               'Healthcare Infrastructure'),
    'Health Transportation Destroyed':                              ('HDX_H_TRANS_DESTR','Health Transportation Destroyed',             'Healthcare Infrastructure'),
    'Health Transportation Damaged':                                ('HDX_H_TRANS_DAMAGE','Health Transportation Damaged',              'Healthcare Infrastructure'),
    'Health Transportation Stolen/Hijacked':                        ('HDX_H_TRANS_STEAL','Health Transportation Stolen/Hijacked',       'Healthcare Infrastructure'),
    'Health Workers Killed':                                        ('HDX_H_W_KILLED',   'Health Workers Killed',                       'Healthcare Personnel'),
    'Health Workers Injured':                                       ('HDX_H_W_INJURED',  'Health Workers Injured',                      'Healthcare Personnel'),
    'Health Workers Kidnapped':                                     ('HDX_H_W_KIDNAP',   'Health Workers Kidnapped',                    'Healthcare Personnel'),
    'Health Workers Arrested':                                      ('HDX_H_W_ARREST',   'Health Workers Arrested',                     'Healthcare Personnel'),
    'Health Workers Threatened':                                    ('HDX_H_W_THREAT',   'Health Workers Threatened',                   'Healthcare Personnel'),
    'Health Workers Assaulted':                                     ('HDX_H_W_ASSAULT',  'Health Workers Assaulted',                    'Healthcare Personnel'),
    'Affects Access to Medicines, Vaccines And Other Health Products': ('HDX_H_MED_ACCESS','Incidents Affecting Medicine/Vaccine Access','Healthcare Access'),
}

health_df = melt_indicators(
    health_raw, health_base, health_indicators,
    subgroup='Health Workers / Health Facilities',
    notes_series='Location: ' + health_raw['Location of Incident'].astype(str),
    prefix='HEALTH'
)
print(f'Health: {len(health_df)} rows')
health_df.head(3)

### 6b. Protection File

In [ ]:
prot_base = get_base(prot_raw, '2020-2026-pse-protection-in-danger', 'SiND ID')

prot_indicators = {
    'Number of Attacks on Camps Reporting Destruction': ('HDX_P_CAMP_DESTR',  'Attacks on Camps — Destruction Reported',  'Protection — Camps'),
    'Number of Attacks on Camps Reporting Damaged':     ('HDX_P_CAMP_DAMAGE', 'Attacks on Camps — Damage Reported',        'Protection — Camps'),
    'Camp Resident Killed':                             ('HDX_P_RES_KILLED',  'Camp Residents Killed',                     'Protection — Casualties'),
    'Camp Resident Injured':                            ('HDX_P_RES_INJURED', 'Camp Residents Injured',                    'Protection — Casualties'),
    'Camp Residents Kidnapped':                         ('HDX_P_RES_KIDNAP',  'Camp Residents Kidnapped',                  'Protection — Casualties'),
    'Camp Residents Arrested':                          ('HDX_P_RES_ARREST',  'Camp Residents Arrested',                   'Protection — Casualties'),
    'Camp Residents Targeted with CRSV':                ('HDX_P_RES_CRSV',    'Camp Residents Targeted with CRSV',         'Protection — CRSV'),
    'Service Provider Killed':                          ('HDX_P_SP_KILLED',   'Service Providers Killed',                  'Protection — Service Providers'),
    'Service Provider Kidnapped':                       ('HDX_P_SP_KIDNAP',   'Service Providers Kidnapped',               'Protection — Service Providers'),
    'Service Provider Arrested':                        ('HDX_P_SP_ARREST',   'Service Providers Arrested',                'Protection — Service Providers'),
    'Service Provider Targeted with CRSV':              ('HDX_P_SP_CRSV',     'Service Providers Targeted with CRSV',      'Protection — CRSV'),
}

prot_df = melt_indicators(
    prot_raw, prot_base, prot_indicators,
    subgroup='Displaced Persons / Camp Residents',
    notes_series='Camp: ' + prot_raw['Camp Name'].astype(str),
    prefix='PROT'
)
print(f'Protection: {len(prot_df)} rows')
prot_df.head(3)

### 6c. Food File

In [ ]:
food_base = get_base(food_raw, '2024-2026-pse-food-systems', 'Sind Event ID')

# Food file has no numeric indicator columns — impact is described in text
# So we keep 1 row per incident and set value = 1 (event occurred)
food_df = food_base.copy()
food_df['indicator_code']      = 'HDX_F_INCIDENT'
food_df['indicator_name']      = 'Food System Incident'
food_df['indicator_category']  = 'Food Security'
food_df['population_subgroup'] = 'General Population / Food System'
food_df['value']               = 1
food_df['unit']                = 'incident count'
food_df['missingness_flag']    = 'NOT_MISSING'
food_df['missingness_type']    = ''
food_df['notes']               = 'Impact: ' + food_raw['Food System Impact'].astype(str)
food_df['data_extracted_date'] = EXTRACTED_DATE
food_df.insert(0, 'record_id', [f'FOOD-{i+1:06d}' for i in range(len(food_df))])
food_df.insert(7, 'date_precision', 'day')
food_df.insert(11, 'geographic_unit_type', 'Admin 1 Region')

print(f'Food: {len(food_df)} rows')
food_df.head(3)

### 6d. Water File

In [ ]:
water_base = get_base(water_raw, '2025-2026-pse-water-systems', 'Sind Event ID')

water_indicators = {
    'Number Of Attacks Damaging/Destroying Water Infrastructure':        ('HDX_W_DAMAGE',   'Water Infrastructure Damaged/Destroyed',          'Water Infrastructure'),
    'Number Of Attacks Where Water Infrastructure Was Contaminated':     ('HDX_W_CONTAM',   'Water Infrastructure Contaminated',                'Water Infrastructure'),
    'Number Of Attacks Where Water Infrastructure Was Looted':           ('HDX_W_LOOTED',   'Water Infrastructure Looted',                      'Water Infrastructure'),
    'Number Of Attacks Where Water Infrastructure Was Obstructed':       ('HDX_W_OBSTRUCT', 'Water Infrastructure Obstructed',                  'Water Infrastructure'),
    'Number Of Attacks Where Water Infrastructure Was Disrupted':        ('HDX_W_DISRUPT',  'Water Infrastructure Disrupted',                   'Water Infrastructure'),
    'Attacks on Persons Collecting Water':                               ('HDX_W_PERSONS',  'Attacks on Persons Collecting Water',              'Water Access — People'),
    'Attacks on People Driving Water Transportation Vehicles':           ('HDX_W_VEHICLES', 'Attacks on Water Transport Vehicle Drivers',       'Water Access — People'),
    'Attacks on Water-Utility Workers':                                  ('HDX_W_WORKERS',  'Attacks on Water-Utility Workers',                 'Water Access — People'),
    'Number Of Incidents Affecting Water-Related Aid Project':           ('HDX_W_AID',      'Incidents Affecting Water Aid Projects',           'Water Infrastructure'),
}

water_df = melt_indicators(
    water_raw, water_base, water_indicators,
    subgroup='General Population / Water Infrastructure',
    notes_series='Infra category: ' + water_raw['Water Infrastructure Category Affected'].astype(str),
    prefix='WATER'
)
print(f'Water: {len(water_df)} rows')
water_df.head(3)

## Step 7: Merge All Sources into One Master Dataset

In [ ]:
FINAL_COLS = [
    'record_id', 'data_source', 'source_file', 'source_version', 'sind_event_id',
    'indicator_code', 'indicator_name', 'indicator_category',
    'date', 'date_precision', 'year',
    'geographic_unit_type', 'admin_1', 'latitude', 'longitude', 'geo_precision',
    'reported_perpetrator', 'reported_perpetrator_name', 'weapon_used',
    'population_subgroup', 'value', 'unit',
    'missingness_flag', 'missingness_type', 'notes', 'data_extracted_date'
]

master = pd.concat([health_df, prot_df, food_df, water_df], ignore_index=True)[FINAL_COLS]

print(f'Master dataset: {len(master)} rows x {len(master.columns)} columns')
print(f'\nMissingness breakdown:')
print(master['missingness_flag'].value_counts())
print(f'\nRows by source:')
print(master['source_file'].value_counts())
master.head(5)

## Step 8: Export to CSV

In [ ]:
# Full dataset
master.to_csv('hdx_master_dataset.csv', index=False)
print(f'Full CSV saved: {os.path.getsize("hdx_master_dataset.csv")/1e6:.1f} MB')

# Gaza Strip only (smaller file for sharing)
gaza_cols = ['record_id','source_file','indicator_code','indicator_name','indicator_category',
             'date','year','admin_1','reported_perpetrator','reported_perpetrator_name',
             'weapon_used','population_subgroup','value','unit','missingness_flag']

master_gaza = master[master['admin_1'] == 'Gaza Strip'][gaza_cols]
master_gaza.to_csv('hdx_master_gaza.csv', index=False)
print(f'Gaza CSV saved: {os.path.getsize("hdx_master_gaza.csv")/1e6:.1f} MB  ({len(master_gaza)} rows)')